In [14]:
import pandas as pd
import ast
import json
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load ATT&CK Data
print("Loading ATT&CK STIX data...")
with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict = {}
name_to_tcode = {}
parent_map = {}

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
        if t_code:
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

# 2. Load SMET Benchmark Data
print("Loading SMET benchmark data...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
with open("id2mitre.json", "r", encoding="utf-8") as f:
    id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
        if mapped_id:
            t_codes.append(parent_map.get(mapped_id, mapped_id))
            
    if t_codes:
        smet_records.append({
            "Description": row["Description"],
            "T_Codes": list(set(t_codes))
        })
smet_df = pd.DataFrame(smet_records)
cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

# 3. Initialize Models
print("Initializing AI Models...")
model_name = "basel/ATTACK-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            output = model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).expand(output.last_hidden_state.size()).float()
            sum_emb = torch.sum(output.last_hidden_state * mask, 1)
            mean_pooled = F.normalize(sum_emb / torch.clamp(mask.sum(1), min=1e-9), p=2, dim=1)
            all_embeddings.append(mean_pooled.cpu())
    return torch.cat(all_embeddings, dim=0)

tech_embeddings = get_embeddings(technique_texts)
bm25_model = BM25Okapi([t.lower().split() for t in technique_texts])
print("Setup Complete.")

Using device: cuda
Loading ATT&CK STIX data...
Loading SMET benchmark data...
Initializing AI Models...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 79620.96it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup Complete.


In [15]:
def map_threat_intel(description, top_k=5, rrf_k=60):
    """
    Takes raw threat intel or CVE descriptions, segments by attack vector,
    and returns top ATT&CK techniques using Hybrid RRF.
    """
    # 1. Chunking
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', description) if len(s.strip()) > 10]
    if not sentences: sentences = [description]
        
    # 2. Semantic
    sentence_embs = get_embeddings(sentences)
    sim_matrix = torch.matmul(sentence_embs, tech_embeddings.T)
    semantic_scores = sim_matrix.max(dim=0).values.numpy()
    semantic_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(semantic_scores)[::-1])}
    
    # 3. Lexical
    bm25_scores = bm25_model.get_scores(description.lower().split())
    lexical_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(bm25_scores)[::-1])}
    
    # 4. RRF Fusion
    rrf_scores = {}
    for idx in range(len(technique_ids)):
        s_rank, l_rank = semantic_ranks[idx], lexical_ranks[idx]
        if bm25_scores[idx] == 0: l_rank = float('inf')
        rrf_scores[idx] = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    
    # 5. Output
    tcode_to_name = {v: k for k, v in name_to_tcode.items()}
    results = []
    for idx in sorted_indices[:top_k]:
        t_code = technique_ids[idx]
        results.append({
            "t_code": t_code,
            "name": tcode_to_name.get(t_code, "Unknown Technique"),
            "rrf_score": round(rrf_scores[idx], 4)
        })
    return results

# Test it!
sample_text = "Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0 (along with 2.12.2, 2.12.3, and 2.3.1), this functionality has been completely removed. Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects."
predictions = map_threat_intel(sample_text, top_k=5)

print(f"INPUT: {sample_text}\n")
for i, p in enumerate(predictions):
    print(f"#{i+1}: [{p['t_code']}] {p['name']} (Score: {p['rrf_score']})")

INPUT: Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0 (along with 2.12.2, 2.12.3, and 2.3.1), this functionality has been completely removed. Note that this vulnerability is specific to log4j-core and does not affect log4net, log4cxx, or other Apache Logging Services projects.

#1: [T1187] Forced Authentication (Score: 0.0313)
#2: [T1212] Exploitation for Credential Access (Score: 0.0286)
#3: [T1568] Dynamic Resolution (Score: 0.0263)
#4: [T1531] Account Access Removal (Score: 0.0259)
#5: [T1207] Rogue Domain Controller (Score: 0.0258)


In [16]:
print("\n" + "="*50)
print("PHASE 1: BEFORE (Standard Bi-Encoder Evaluation)")
print("="*50)

# Embed full CVE texts
cve_full_embeddings = get_embeddings(cve_texts)
similarity_matrix = torch.matmul(cve_full_embeddings, tech_embeddings.T)

hits_at_1, hits_at_5, hits_at_10 = 0, 0, 0
n_cves = len(cve_texts)

for i in range(n_cves):
    true_labels = set(cve_ground_truths[i])
    top_10_indices = torch.topk(similarity_matrix[i], k=10).indices.tolist()
    top_10_preds = [technique_ids[idx] for idx in top_10_indices]
    
    if top_10_preds[0] in true_labels: hits_at_1 += 1
    if len(true_labels.intersection(set(top_10_preds[:5]))) > 0: hits_at_5 += 1
    if len(true_labels.intersection(set(top_10_preds))) > 0: hits_at_10 += 1

print(f"Standard Hit Rate@1:  {(hits_at_1 / n_cves) * 100:.2f}%")
print(f"Standard Hit Rate@5:  {(hits_at_5 / n_cves) * 100:.2f}%")
print(f"Standard Hit Rate@10: {(hits_at_10 / n_cves) * 100:.2f}%")


PHASE 1: BEFORE (Standard Bi-Encoder Evaluation)
Standard Hit Rate@1:  25.17%
Standard Hit Rate@5:  55.30%
Standard Hit Rate@10: 68.54%


In [17]:
print("\n" + "="*50)
print("PHASE 2: AFTER (Chunked Hybrid RRF Evaluation)")
print("="*50)


hits_at_1, hits_at_5, hits_at_10 = 0, 0, 0
n_cves = len(cve_texts)

hits_at_1_new, hits_at_5_new, hits_at_10_new = 0, 0, 0
rrf_k = 60

for i in tqdm(range(n_cves), desc="Evaluating Hybrid Pipeline"):
    true_labels = set(cve_ground_truths[i])
    text = cve_texts[i]
    
    # 1. Chunking (Simulating Attack Vectors without heavy SRL)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', text) if len(s.strip()) > 10]
    if not sentences: sentences = [text]
        
    # 2. Semantic Ranking (Max pooling across sentence chunks)
    sentence_embs = get_embeddings(sentences)
    sim_matrix = torch.matmul(sentence_embs, tech_embeddings.T)
    semantic_scores = sim_matrix.max(dim=0).values.numpy() # The magic step!
    semantic_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(semantic_scores)[::-1])}
    
    # 3. Lexical Ranking (BM25 on full text)
    bm25_scores = bm25_model.get_scores(text.lower().split())
    lexical_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(bm25_scores)[::-1])}
    
    # 4. Reciprocal Rank Fusion
    rrf_scores = {}
    for idx in range(len(technique_ids)):
        s_rank, l_rank = semantic_ranks[idx], lexical_ranks[idx]
        if bm25_scores[idx] == 0: l_rank = float('inf')
        rrf_scores[idx] = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    top_10_preds = [technique_ids[idx] for idx in sorted_indices[:10]]
    
    if top_10_preds[0] in true_labels: hits_at_1_new += 1
    if len(true_labels.intersection(set(top_10_preds[:5]))) > 0: hits_at_5_new += 1
    if len(true_labels.intersection(set(top_10_preds))) > 0: hits_at_10_new += 1

print(f"Hybrid Hit Rate@1:  {(hits_at_1_new / n_cves) * 100:.2f}%")
print(f"Hybrid Hit Rate@5:  {(hits_at_5_new / n_cves) * 100:.2f}%")
print(f"Hybrid Hit Rate@10: {(hits_at_10_new / n_cves) * 100:.2f}%")


PHASE 2: AFTER (Chunked Hybrid RRF Evaluation)


Evaluating Hybrid Pipeline: 100%|██████████| 302/302 [00:05<00:00, 55.29it/s]

Hybrid Hit Rate@1:  35.76%
Hybrid Hit Rate@5:  66.56%
Hybrid Hit Rate@10: 78.15%


In [1]:
# ==============================================================================
# CELL 1: Setup & Data Extraction
# ==============================================================================
import pandas as pd
import ast
import json
import torch
import torch.nn.functional as F
import numpy as np
import pickle
import re
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from scipy.special import softmax
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load ATT&CK STIX Data
print("Loading ATT&CK v16.1 Data...")
with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict, name_to_tcode, parent_map = {}, {}, {}

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
        if t_code:
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id = obj.get("source_ref")
        parent_id = obj.get("target_ref")
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        if sub_tcode and parent_tcode: parent_map[sub_tcode] = parent_tcode

parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

# 2. Load SMET Benchmark
print("Loading SMET Benchmark...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
with open("id2mitre.json", "r", encoding="utf-8") as f: id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    try: tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except: tech_names = []
        
    t_codes = []
    for name in tech_names:
        mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
        if mapped_id: t_codes.append(parent_map.get(mapped_id, mapped_id))
            
    if t_codes:
        smet_records.append({"Description": row["Description"], "T_Codes": list(set(t_codes))})

smet_df = pd.DataFrame(smet_records)
cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

print(f"Loaded {len(cve_texts)} valid SMET CVEs.")

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading ATT&CK v16.1 Data...
Loading SMET Benchmark...
Loaded 302 valid SMET CVEs.


In [6]:
# ==============================================================================
# CELL 2: Initialize AI Models & Search Indices
# ==============================================================================
print("Initializing ATT&CK-BERT...")
model_name = "basel/ATTACK-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            output = model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).expand(output.last_hidden_state.size()).float()
            sum_emb = torch.sum(output.last_hidden_state * mask, 1)
            mean_pooled = F.normalize(sum_emb / torch.clamp(mask.sum(1), min=1e-9), p=2, dim=1)
            all_embeddings.append(mean_pooled.cpu())
    return torch.cat(all_embeddings, dim=0)

print("Building Technique Embeddings...")
tech_embeddings = get_embeddings(technique_texts)

print("Building BM25 Lexical Index...")
bm25_model = BM25Okapi([t.lower().split() for t in technique_texts])

print("Loading SMET LR Classifier & Dictionaries...")
LR_model = pickle.load(open("LR_ATT&CK_model_V2.pkl", 'rb'))
with open('id2ATT&CK_V2.json', 'r', encoding='utf-8') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

print("✅ All Models Ready.")

Initializing ATT&CK-BERT...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34750.26it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building Technique Embeddings...
Building BM25 Lexical Index...
Loading SMET LR Classifier & Dictionaries...
✅ All Models Ready.


c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [17]:
# ==============================================================================
# CELL 3: Optimized Mapping Engines
# ==============================================================================
from sentence_transformers import SentenceTransformer

# 1. Initialize the SentenceTransformer ONCE here, not in the function
print("Loading SentenceTransformer for SMET Logic...")
smet_emb_model = SentenceTransformer("basel/ATTACK-BERT", local_files_only=True)

# --- Method 1: Standard Bi-Encoder ---
def predict_m1_baseline(text, top_k=10):
    # Use the model we already loaded in Cell 2
    emb = get_embeddings([text])
    sim_scores = torch.matmul(emb, tech_embeddings.T)[0].numpy()
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    return [technique_ids[idx] for idx in top_indices]

# --- Method 2: Chunked Hybrid RRF ---
def predict_m2_hybrid_rrf(text, top_k=10, rrf_k=60):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', text) if len(s.strip()) > 10]
    if not sentences: sentences = [text]
    
    # Use the global 'model' from Cell 2
    sentence_embs = get_embeddings(sentences)
    sim_matrix = torch.matmul(sentence_embs, tech_embeddings.T)
    semantic_scores = sim_matrix.max(dim=0).values.numpy()
    semantic_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(semantic_scores)[::-1])}
    
    bm25_scores = bm25_model.get_scores(text.lower().split())
    lexical_ranks = {idx: r + 1 for r, idx in enumerate(np.argsort(bm25_scores)[::-1])}
    
    rrf_scores = {}
    for idx in range(len(technique_ids)):
        s_rank, l_rank = semantic_ranks[idx], lexical_ranks[idx]
        if bm25_scores[idx] == 0: l_rank = float('inf')
        rrf_scores[idx] = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [technique_ids[idx] for idx in sorted_idx[:top_k]]

# --- Method 3: True SMET Architecture ---

# ==============================================================================
# FIXED: Method 3 with Name-to-TCode Mapping and Parent Rollup
# ==============================================================================

def predict_m3_true_smet(text, top_k=10):
    # 1. Chunking logic consistent with SMET architecture
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', text) if len(s.strip()) > 10]
    attack_vectors = sentences + [text]
    
    all_predictions = []
    for av in attack_vectors:
        if not av.strip(): continue
        
        # Get features using the specialized SMET embedding model
        emb = smet_emb_model.encode(av, show_progress_bar=False)
        
        # Run Logistic Regression classifier
        dec = LR_model.decision_function([emb])
        probs = softmax(dec)[0]
        
        for i in range(len(dec[0])):
            try:
                # The LR model outputs an index (i) 
                # We map index -> name -> T-code -> Parent T-code
                tech_name = id2label[i]
                
                # Use name_to_tcode (from Cell 1) for the most reliable mapping
                raw_tcode = name_to_tcode.get(tech_name)
                
                if raw_tcode:
                    # Roll up to parent (Txxxx.yyy -> Txxxx) to match ground truth
                    parent_tc = parent_map.get(raw_tcode, raw_tcode)
                    all_predictions.append((parent_tc, probs[i]))
            except KeyError:
                continue
            
    # 2. Consolidate: Take the maximum probability found across all segments
    consolidated = {}
    for t_code, prob in all_predictions:
        if t_code not in consolidated or prob > consolidated[t_code]:
            consolidated[t_code] = prob
            
    # 3. Rank and return top K T-codes
    sorted_items = sorted(consolidated.items(), key=lambda x: x[1], reverse=True)
    return [item[0] for item in sorted_items[:top_k]]

Loading SentenceTransformer for SMET Logic...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 68708.14it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ==============================================================================
# CELL 4: Master Evaluation & Scoreboard
# ==============================================================================
print("Starting Master Evaluation Loop...")

# Tracking dictionaries for Hits@1, Hits@5, Hits@10
hits = {
    "M1": {1: 0, 5: 0, 10: 0},
    "M2": {1: 0, 5: 0, 10: 0},
    "M3": {1: 0, 5: 0, 10: 0}
}

n_cves = len(cve_texts)

for i in tqdm(range(n_cves), desc="Benchmarking 3 Methods"):
    true_labels = set(cve_ground_truths[i])
    text = cve_texts[i]
    
    # Run Predictions
    preds_m1 = predict_m1_baseline(text, top_k=10)
    preds_m2 = predict_m2_hybrid_rrf(text, top_k=10)
    # preds_m3 = predict_m3_true_smet(text, top_k=10)
    
    # Helper to calculate hits
    def calc_hits(method_key, preds):
        if len(preds) > 0 and preds[0] in true_labels: 
            hits[method_key][1] += 1
        if len(true_labels.intersection(set(preds[:5]))) > 0: 
            hits[method_key][5] += 1
        if len(true_labels.intersection(set(preds))) > 0: 
            hits[method_key][10] += 1

    calc_hits("M1", preds_m1)
    calc_hits("M2", preds_m2)
    # calc_hits("M3", preds_m3)

# Print Final Scoreboard
print("\n" + "═"*75)
print(f"{'THE MASTER SCOREBOARD: CVE MAPPING ARCHITECTURES':^75}")
print("═"*75)
print(f"{'Methodology':<35} | {'Hit Rate@1':<10} | {'Hit Rate@5':<10} | {'Hit Rate@10':<10}")
print("─"*75)

methods = [
    ("M1", "1. Standard Bi-Encoder (Baseline)"),
    ("M2", "2. Chunked Hybrid RRF (Your Engine)"),
    ("M3", "3. True SMET (LR Classifier)")
]

for key, name in methods:
    h1 = (hits[key][1] / n_cves) * 100
    h5 = (hits[key][5] / n_cves) * 100
    h10 = (hits[key][10] / n_cves) * 100
    print(f"{name:<35} | {h1:>8.2f}%  | {h5:>8.2f}%  | {h10:>9.2f}%")
print("═"*75)

Starting Master Evaluation Loop...


Benchmarking 3 Methods: 100%|██████████| 302/302 [00:26<00:00, 11.44it/s]


═══════════════════════════════════════════════════════════════════════════
             THE MASTER SCOREBOARD: CVE MAPPING ARCHITECTURES              
═══════════════════════════════════════════════════════════════════════════
Methodology                         | Hit Rate@1 | Hit Rate@5 | Hit Rate@10
───────────────────────────────────────────────────────────────────────────
1. Standard Bi-Encoder (Baseline)   |    25.17%  |    55.30%  |     68.54%
2. Chunked Hybrid RRF (Your Engine) |    35.76%  |    66.56%  |     78.15%
3. True SMET (LR Classifier)        |     0.00%  |     0.00%  |      0.00%
═══════════════════════════════════════════════════════════════════════════


In [19]:
import pandas as pd
import ast
import json
import re
import pickle
import numpy as np
from scipy.special import softmax
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print("Loading SMET Models and Classifiers...")

# 1. Load the Models exactly as SMET does
emb_model = SentenceTransformer("basel/ATTACK-BERT")
LR_model = pickle.load(open("LR_ATT&CK_model_V2.pkl", 'rb'))

# 2. Load the Label Dictionaries
id2mitre = json.load(open('id2mitre.json', 'r'))
id2label_raw = json.load(open('id2ATT&CK_V2.json', 'r'))
id2label = {int(i): v for i, v in id2label_raw.items()}

# ==========================================
# Benchmark Data Loading (From previous step)
# ==========================================
print("Loading ATT&CK STIX data & Benchmark Data...")
with open(r"OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

name_to_tcode = {}
parent_map = {}

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = next((ref.get("external_id") for ref in obj.get("external_references", []) if ref.get("source_name") == "mitre-attack"), None)
        if t_code:
            name_to_tcode[obj.get("name")] = t_code

for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id, parent_id = obj.get("source_ref"), obj.get("target_ref")
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")
smet_records = []
for _, row in smet_raw.iterrows():
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        mapped_id = next((k for k, v in id2mitre.items() if name in v), name_to_tcode.get(name))
        if mapped_id:
            t_codes.append(parent_map.get(mapped_id, mapped_id))
            
    if t_codes:
        smet_records.append({
            "Description": row["Description"],
            "T_Codes": list(set(t_codes))
        })
        
smet_df = pd.DataFrame(smet_records)
cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

# ==============================================================================
# The Exact SMET Prediction Logic
# ==============================================================================
def predict_techniques_smet(emb, clf, id2label_map, id2mitre_map):
    dec = clf.decision_function([emb])
    out = softmax(dec)[0]
    
    mapped_results = []
    for i in range(len(dec[0])):
        try:
            # id2label actually returns the T-Code (e.g., "T1068")
            actual_t_code = id2label_map[i] 
            
            # id2mitre actually returns the Name (e.g., "Exploitation for Privilege Escalation")
            name_match = id2mitre_map[actual_t_code]
            actual_name = name_match[0] if isinstance(name_match, list) else name_match
            
            # Append in the correct order: (t_code, name, probability)
            mapped_results.append((actual_t_code, actual_name, out[i]))
        except KeyError:
            continue
            
    return sorted(mapped_results, key=lambda x: x[2], reverse=True)

def map_cve_smet_way(description, top_k=10): # Changed to 10 to support R@10
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|;\s+', description) if len(s.strip()) > 10]
    attack_vectors = sentences + [description]
    
    all_predictions = []
    for av in attack_vectors:
        if not av.strip():
            continue
        emb = emb_model.encode(av)
        preds = predict_techniques_smet(emb, LR_model, id2label, id2mitre)
        all_predictions.extend(preds)
        
    consolidated = {}
    for t_code, name, prob in all_predictions:
        if t_code not in consolidated or prob > consolidated[t_code]['prob']:
            consolidated[t_code] = {'name': name, 'prob': prob}
            
    final_sorted = sorted(consolidated.items(), key=lambda item: item[1]['prob'], reverse=True)
    
    results = []
    for t_code, data in final_sorted[:top_k]:
        results.append({
            "t_code": t_code,
            "name": data['name'],
            "confidence": round(data['prob'], 4)
        })
        
    return results

# ==========================================
# Full Dataset Evaluation Loop
# ==========================================
print("\nSetup Complete. Running full dataset evaluation...")

smet_metrics = {'R@1': [], 'R@5': [], 'R@10': []}

for idx, cve_desc in enumerate(tqdm(cve_texts, desc="Evaluating CVEs")):
    ground_truth = cve_ground_truths[idx]
    if not ground_truth:
        continue

    # Get Top 10 predictions using your SMET architecture
    smet_preds = map_cve_smet_way(cve_desc, top_k=10)
    
    # Extract just the T-codes
    s_tcodes = [p['t_code'] for p in smet_preds]

    # Calculate Recall @ K
    for k in [1, 5, 10]:
        s_hits = len(set(s_tcodes[:k]).intersection(set(ground_truth)))
        smet_metrics[f'R@{k}'].append(s_hits / len(ground_truth))

# Print Final Averaged Results
print("\n" + "="*50)
print(f"EVALUATION RESULTS (Tested on {len(smet_metrics['R@1'])} CVEs)")
print("="*50)

print("--- Pure SMET Architecture ---")
print(f"Recall@1:  {np.mean(smet_metrics['R@1']):.4f}")
print(f"Recall@5:  {np.mean(smet_metrics['R@5']):.4f}")
print(f"Recall@10: {np.mean(smet_metrics['R@10']):.4f}")
print("="*50)

Loading SMET Models and Classifiers...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25079.37it/s]
MPNetModel LOAD REPORT from: basel/ATTACK-BERT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loading ATT&CK STIX data & Benchmark Data...

Setup Complete. Running full dataset evaluation...


Evaluating CVEs: 100%|██████████| 302/302 [00:17<00:00, 17.49it/s]


EVALUATION RESULTS (Tested on 302 CVEs)
--- Pure SMET Architecture ---
Recall@1:  0.2707
Recall@5:  0.5944
Recall@10: 0.7398
